# Random Tanh RNN with Log-Normal Weights

Nonlinear (tanh) random-network simulation with **lognormal** connectivity,
mirroring the analysis from `sandbox/linear_network_timescales/random_linear_networks.ipynb`.

For each gain $g$ we:
1. Simulate $\tau\dot x = -x + g W \tanh(x) + \eta(t)$ with Euler–Maruyama
2. Project activity onto eigenvectors of $W$
3. Fit exponential decays to neuron and eigenmode autocorrelations
4. Compare fitted timescales to linear-theory predictions $\tau_k = -1/\mathrm{Re}(\lambda_k)$

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from typing import Tuple
from numpy.typing import NDArray

SEED = 42
rng_global = np.random.default_rng(SEED)

plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print(f"Random seed: {SEED}")

## Helper functions

In [ ]:
def generate_W_lognormal(n: int, rng: np.random.Generator) -> NDArray:
    """|W_ij| ~ LogNormal(0,1), random +/- signs, scaled by 1/sqrt(N)."""
    magnitudes = rng.lognormal(mean=0.0, sigma=1.0, size=(n, n))
    signs = rng.choice([-1.0, 1.0], size=(n, n))
    return signs * magnitudes / np.sqrt(n)


def get_jacobian_eigenvalues(eigenvalues_W: NDArray, g: float, tau: float) -> NDArray:
    """lambda_M = (1/tau)(g * lambda_W - 1)"""
    return (g * eigenvalues_W - 1.0) / tau


def get_effective_timescales(eigenvalues_M: NDArray) -> NDArray:
    """tau_eff = -1 / Re(lambda_M) for stable modes."""
    real_parts = np.real(eigenvalues_M)
    stable_mask = real_parts < 0
    return -1.0 / real_parts[stable_mask]


def get_stable_indices(eigenvalues_M: NDArray) -> NDArray:
    return np.where(np.real(eigenvalues_M) < 0)[0]


def project_onto_eigenvectors(activity: NDArray, eigenvectors: NDArray) -> NDArray:
    V_inv = np.linalg.inv(eigenvectors)
    return activity @ V_inv.T


def compute_autocorrelation(z_k: NDArray, max_lag: int):
    """
    ACF handling complex eigenvectors correctly.
    Real mode:    standard ACF of Re(z_k)
    Complex mode: |E[z(t+tau) z*(t)]| / C(0)
    Returns (acf, is_complex).
    """
    n = len(z_k)
    is_cx = np.iscomplexobj(z_k) and np.abs(np.imag(z_k)).max() > 1e-10
    z_centered = z_k - np.mean(z_k)
    fft_len = 2 ** int(np.ceil(np.log2(2 * n - 1)))
    Z = np.fft.fft(z_centered, fft_len)
    acf_full = np.fft.ifft(Z * np.conj(Z))
    acf = np.abs(acf_full[:max_lag + 1]) if is_cx else np.real(acf_full[:max_lag + 1])
    if acf[0] > 0:
        acf = acf / acf[0]
    return acf, is_cx


def fit_exponential_timescale(
    autocorr: NDArray, dt: float, fit_range: Tuple[int, int] = (1, None)
) -> float:
    start, end = fit_range[0], fit_range[1] or len(autocorr)
    lags = np.arange(start, end)
    ac = autocorr[start:end]
    positive_mask = ac > 0.01
    if positive_mask.sum() < 5:
        return np.nan
    slope, *_ = stats.linregress(lags[positive_mask] * dt, np.log(ac[positive_mask]))
    return np.nan if slope >= 0 else -1.0 / slope


def simulate_nonlinear_ode(
    W: NDArray, g: float, tau: float, dt: float,
    duration: float, noise_std: float, rng: np.random.Generator,
) -> NDArray:
    """Simulate tau*dx/dt = -x + g*W*tanh(x) + noise (Euler–Maruyama)."""
    n = W.shape[0]
    n_steps = int(duration / dt)
    activity = np.zeros((n_steps, n))
    x = rng.standard_normal(n) * 0.1
    dt_tau = dt / tau
    sqrt_dt_tau = np.sqrt(dt_tau)
    gW = g * W
    for t in range(n_steps):
        activity[t] = x
        noise = rng.standard_normal(n) * noise_std * sqrt_dt_tau
        x = x + dt_tau * (-x + gW @ np.tanh(x)) + noise
        if t % 10000 == 0 and t > 0:
            print(f"  Step {t}/{n_steps}")
    return activity


def zscore_rows(arr):
    return (arr - arr.mean(axis=1, keepdims=True)) / (arr.std(axis=1, keepdims=True) + 1e-10)


print("Helpers defined.")

## Sweep config

In [ ]:
g_values = [0.3, 0.5, 0.8, 0.95, 0.99, 1.2, 1.5]

sim_config = {
    "n_neurons": 1000,
    "tau":       1.0,
    "dt":        0.1,
    "duration":  5000.0,
    "noise_std": 0.005,
}

max_lag = 300  # in steps

rng_sweep = np.random.default_rng(0)
n_neurons = sim_config["n_neurons"]
tau       = sim_config["tau"]
dt        = sim_config["dt"]

W = generate_W_lognormal(n_neurons, rng_sweep)
eigenvalues_W, eigenvectors = np.linalg.eig(W)
print(f"W shape: {W.shape}")
print(f"Spectral radius of W: {np.max(np.abs(eigenvalues_W)):.3f}")

## Run simulations across g

In [ ]:
results = []

for g in g_values:
    print(f"\n── g = {g} ──────────────────────────")
    duration = sim_config["duration"]

    eigs_M     = get_jacobian_eigenvalues(eigenvalues_W, g, tau)
    stab_idx   = get_stable_indices(eigs_M)
    tau_th_all = get_effective_timescales(eigs_M)

    discard = int(100 / dt)

    # Nonlinear simulation
    print("  Simulating nonlinear ...")
    act = simulate_nonlinear_ode(
        W, g, tau, dt, duration, sim_config["noise_std"], rng_sweep
    )[discard:]
    proj = project_onto_eigenvectors(act, eigenvectors)

    # Fit neuron autocorrelations
    print("  Fitting neuron timescales ...")
    tau_neu = np.full(n_neurons, np.nan)
    for i in range(n_neurons):
        acf, _ = compute_autocorrelation(act[:, i], max_lag)
        tau_neu[i] = fit_exponential_timescale(acf, dt, fit_range=(1, 30))

    # Fit eigenmode autocorrelations (stable modes only)
    print("  Fitting eigenmode timescales ...")
    tau_mode = np.full(len(stab_idx), np.nan)
    for j, midx in enumerate(stab_idx):
        acf, _ = compute_autocorrelation(proj[:, midx], max_lag)
        tau_mode[j] = fit_exponential_timescale(acf, dt, fit_range=(1, 30))

    valid_neu  = np.isfinite(tau_neu)  & (tau_neu  > 0)
    valid_mode = np.isfinite(tau_mode) & (tau_mode > 0)

    results.append(dict(
        g           = g,
        tau_th_all  = tau_th_all,
        stab_idx    = stab_idx,
        tau_neu     = tau_neu,
        tau_mode    = tau_mode,
        valid_neu   = valid_neu,
        valid_mode  = valid_mode,
        activity    = act,
        projections = proj,
    ))
    print(f"  Valid fits: neurons {valid_neu.sum()}/{n_neurons}, modes {valid_mode.sum()}/{len(stab_idx)}")

print("\nSweep complete.")

## Single-g diagnostic: traces, heatmaps, ACFs

Pick one representative `g` for detailed plots (matching Part 3 of reference notebook).

In [ ]:
g_show = 1.5
res = next(r for r in results if r["g"] == g_show)

act_show  = res["activity"]
proj_show = res["projections"]
stab_idx_show   = res["stab_idx"]
tau_th_show     = res["tau_th_all"]
tau_neu_show    = res["tau_neu"]
tau_mode_show   = res["tau_mode"]
valid_neu_show  = res["valid_neu"]
valid_mode_show = res["valid_mode"]

eigs_M_show = get_jacobian_eigenvalues(eigenvalues_W, g_show, tau)

_title_suffix = rf'$g={g_show}$,  $\tau_0={tau}$,  $N={n_neurons}$  [tanh, lognormal $W$]'

### Sample traces

In [ ]:
n_show = 10
rng_display = np.random.default_rng(0)

neuron_display_idx = rng_display.choice(np.where(valid_neu_show)[0], size=n_show, replace=False)
valid_mode_local   = np.where(valid_mode_show)[0]
mode_display_local = rng_display.choice(valid_mode_local, size=n_show, replace=False)
mode_display_global = stab_idx_show[mode_display_local]

plot_steps = min(int(500 / dt), act_show.shape[0])
time_win   = np.arange(plot_steps) * dt
colors_show = plt.cm.tab10(np.arange(n_show) / 10)
offset_scale = 4.0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for i in range(n_show):
    trace = act_show[:plot_steps, neuron_display_idx[i]]
    trace_norm = trace / (trace.std() + 1e-10)
    ax.plot(time_win, trace_norm + i * offset_scale,
            color=colors_show[i], linewidth=0.8,
            label=f'Neuron {neuron_display_idx[i]}')
ax.set_xlabel('Time'); ax.set_yticks([])
ax.set_title('Neural activity')
ax.legend(loc='upper right', fontsize=8); ax.grid(True, alpha=0.2)

ax = axes[1]
for i in range(n_show):
    gidx   = mode_display_global[i]
    lidx   = mode_display_local[i]
    tau_th = tau_th_show[lidx]
    eig_val = eigs_M_show[gidx]
    is_cx   = np.abs(np.imag(eig_val)) > 1e-8

    trace = np.sqrt(2) * np.abs(proj_show[:plot_steps, gidx]) if is_cx \
            else np.abs(np.real(proj_show[:plot_steps, gidx]))
    label = f'Mode {gidx}  (τ={tau_th:.1f})'
    trace_norm = trace / (trace.std() + 1e-10)
    ax.plot(time_win, trace_norm + i * offset_scale,
            color=colors_show[i], linewidth=0.8, label=label)

ax.set_xlabel('Time'); ax.set_yticks([])
ax.set_title(r'Eigenmode  $\sqrt{2}|z_k|$')
ax.legend(loc='upper right', fontsize=8); ax.grid(True, alpha=0.2)

fig.suptitle(f'Sample traces  —  {_title_suffix}', fontsize=13)
plt.tight_layout(); plt.show()

### Population heatmaps

In [ ]:
n_rows_show = 200
plot_steps_hm = min(int(500 / dt), act_show.shape[0])
time_win_hm   = np.arange(plot_steps_hm) * dt

row_idx = np.arange(n_rows_show)
N_rows  = len(row_idx)

raw_neurons = act_show[:plot_steps_hm, row_idx].T
raw_modes   = np.real(proj_show[:plot_steps_hm, row_idx]).T

data_neurons = zscore_rows(raw_neurons)
data_modes   = zscore_rows(raw_modes)

kw = dict(aspect='auto', vmin=-2.5, vmax=2.5, cmap='RdBu_r',
          extent=[0, time_win_hm[-1], N_rows - 0.5, -0.5],
          interpolation='none')
ytick_pos    = np.linspace(0, N_rows - 1, 6, dtype=int)
ytick_labels = [str(p) for p in ytick_pos]

fig, axes = plt.subplots(1, 2, figsize=(14, 8), constrained_layout=True)

ax = axes[0]
im = ax.imshow(data_neurons, **kw)
ax.set_xlabel('Time'); ax.set_ylabel('Neuron index')
ax.set_title(f'Neural activity  (showing {N_rows}/{n_neurons})')
ax.set_yticks(ytick_pos); ax.set_yticklabels(ytick_labels)
fig.colorbar(im, ax=ax, label='z-score', shrink=0.6)

ax = axes[1]
im = ax.imshow(data_modes, **kw)
ax.set_xlabel('Time'); ax.set_ylabel('Mode index')
ax.set_title(r'Eigenmode projections  $\mathrm{Re}(z_k)$  ' + f'(showing {N_rows}/{n_neurons})')
ax.set_yticks(ytick_pos); ax.set_yticklabels(ytick_labels)
fig.colorbar(im, ax=ax, label='z-score', shrink=0.6)

fig.suptitle(f'Full population heatmaps  —  {_title_suffix}', fontsize=13)
plt.show()

### Per-unit autocorrelations with fits

In [ ]:
n_show_ac = 8
lags_s = np.arange(max_lag + 1) * dt

fig, axes = plt.subplots(n_show_ac, 2, figsize=(14, 3.5 * n_show_ac), sharex=True)
axes[0, 0].set_title('Neural activity', fontsize=12)
axes[0, 1].set_title('Eigenmode projections', fontsize=12)

for i in range(n_show_ac):
    # Neuron ACF
    ax   = axes[i, 0]
    nidx = neuron_display_idx[i]
    acf, _ = compute_autocorrelation(act_show[:, nidx], max_lag)
    tau_fit = tau_neu_show[nidx]

    ax.semilogy(lags_s, np.maximum(acf, 1e-6), color='steelblue', lw=1.5, label='ACF')
    if np.isfinite(tau_fit):
        ax.semilogy(lags_s, np.exp(-lags_s / tau_fit), 'k--', lw=1.5,
                    label=rf'Fitted  $\tau={tau_fit:.1f}$')
    ax.set_ylabel('Autocorrelation')
    ax.grid(True, alpha=0.3); ax.legend(fontsize=9, loc='upper right')
    ax.annotate(f'Neuron {nidx}', xy=(0.03, 0.05), xycoords='axes fraction',
                fontsize=9, bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.8))
    ax.set_ylim([1e-2, 1.5])

    # Eigenmode ACF
    ax   = axes[i, 1]
    lidx = mode_display_local[i]
    gidx = mode_display_global[i]
    acf, is_cx = compute_autocorrelation(proj_show[:, gidx], max_lag)
    tau_th  = tau_th_show[lidx]
    tau_fit = tau_mode_show[lidx]

    ax.semilogy(lags_s, np.maximum(acf, 1e-6), color='steelblue', lw=1.5,
                label=r'$|\mathrm{ACF}|$' if is_cx else 'ACF')
    if np.isfinite(tau_fit):
        ax.semilogy(lags_s, np.exp(-lags_s / tau_fit), 'k--', lw=1.5,
                    label=rf'Fitted  $\tau={tau_fit:.1f}$')
    ax.semilogy(lags_s, np.exp(-lags_s / tau_th), 'r--', lw=1.5,
                label=rf'Theory  $\tau={tau_th:.1f}$')
    ax.set_ylabel('Autocorrelation')
    ax.grid(True, alpha=0.3); ax.legend(fontsize=9, loc='upper right')
    suffix = '\u2102' if is_cx else '\u211d'
    ax.annotate(f'[{suffix}] Mode {gidx}', xy=(0.03, 0.05), xycoords='axes fraction',
                fontsize=9, bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.8))
    ax.set_ylim([1e-4, 1.5])

axes[-1, 0].set_xlabel('Lag (time units)')
axes[-1, 1].set_xlabel('Lag (time units)')
fig.suptitle(f'Autocorrelations  —  {_title_suffix}', fontsize=13)
plt.tight_layout(); plt.show()

### Theory vs simulation (single g)

In [ ]:
tau_theory_paired = tau_th_show[valid_mode_show]
tau_mode_valid    = tau_mode_show[valid_mode_show]
tau_neuron_valid  = tau_neu_show[valid_neu_show]

all_vals = np.concatenate([tau_theory_paired, tau_neuron_valid, tau_mode_valid])
bins = np.logspace(
    np.log10(np.percentile(all_vals, 1)),
    np.log10(np.percentile(all_vals, 99)),
    50,
)

fig, axes = plt.subplots(2, 2, figsize=(13, 11))

ax = axes[0, 0]
ax.hist(tau_theory_paired, bins=bins, alpha=0.55, density=True, color='tab:blue', edgecolor='white',
        label=f'Linear Theory  (n={len(tau_theory_paired)})')
ax.hist(tau_neuron_valid, bins=bins, alpha=0.55, density=True, color='tab:orange', edgecolor='white',
        label=f'Neurons  (n={len(tau_neuron_valid)})')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel(r'$\tau_{\rm eff}$'); ax.set_ylabel('Density')
ax.set_title('Linear theory vs Neurons')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.hist(tau_theory_paired, bins=bins, alpha=0.55, density=True, color='tab:blue', edgecolor='white',
        label=f'Linear Theory  (n={len(tau_theory_paired)})')
ax.hist(tau_mode_valid, bins=bins, alpha=0.55, density=True, color='tab:green', edgecolor='white',
        label=f'Eigenmodes  (n={len(tau_mode_valid)})')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel(r'$\tau_{\rm eff}$'); ax.set_ylabel('Density')
ax.set_title('Linear theory vs Jacobian eigenmodes')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1, 0]
n_min = min(len(tau_theory_paired), len(tau_neuron_valid))
ax.scatter(np.sort(tau_theory_paired)[:n_min], np.sort(tau_neuron_valid)[:n_min],
           s=8, alpha=0.3, color='tab:orange')
v0 = min(tau_theory_paired.min(), tau_neuron_valid.min())
v1 = max(tau_theory_paired.max(), tau_neuron_valid.max())
ax.plot([v0, v1], [v0, v1], 'r--', lw=2, label='Identity')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel(r'Theory $\tau$ (sorted)'); ax.set_ylabel(r'Fitted $\tau$ — neurons (sorted)')
ax.set_title('Rank-rank: Theory vs Neuron fits')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_aspect('equal')

ax = axes[1, 1]
ax.scatter(tau_theory_paired, tau_mode_valid, s=8, alpha=0.3, color='tab:green')
v0 = min(tau_theory_paired.min(), tau_mode_valid.min())
v1 = max(tau_theory_paired.max(), tau_mode_valid.max())
ax.plot([v0, v1], [v0, v1], 'r--', lw=2, label='Identity')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel(r'Theory $\tau$'); ax.set_ylabel(r'Fitted $\tau$ — eigenmodes')
ax.set_title('1-to-1: Theory vs Eigenmode fits')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_aspect('equal')

fig.suptitle(f'Theory vs simulation  —  {_title_suffix}', fontsize=13)
plt.tight_layout(); plt.show()

## Timescale distributions across g values

Grid plot matching Part 4 of the reference notebook: one row per `g`, columns for theory-vs-neurons and theory-vs-eigenmodes.

In [ ]:
n_g   = len(results)
n_col = 2
col_titles = [
    'Nonlinear network — linear theory vs neurons',
    'Nonlinear network — linear theory vs Jacobian eigenmodes',
]

fig, axes = plt.subplots(
    n_g, n_col,
    figsize=(5 * n_col, 3.5 * n_g),
    sharex='col', sharey='col',
    constrained_layout=True,
)
if n_g == 1:
    axes = axes[np.newaxis, :]

for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=11)

def _hist_panel(ax, tau_theory, tau_fitted, color, label_fit, bins):
    if len(tau_theory) == 0 or len(tau_fitted) == 0:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                transform=ax.transAxes, fontsize=11, color='gray')
        return
    ax.hist(tau_theory, bins=bins, alpha=0.55, density=True,
            color='tab:blue', edgecolor='white', label='Linear Theory')
    ax.hist(tau_fitted, bins=bins, alpha=0.55, density=True,
            color=color, edgecolor='white', label=label_fit)
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, loc='upper right')

for row, res in enumerate(results):
    g             = res['g']
    tau_th_all    = res['tau_th_all']
    valid_mode_r  = res['valid_mode']
    valid_neu_r   = res['valid_neu']

    tau_th_paired = tau_th_all[valid_mode_r]

    all_v = np.concatenate([
        tau_th_paired,
        res['tau_neu'][valid_neu_r],
        res['tau_mode'][valid_mode_r],
    ])
    if len(all_v) == 0:
        continue
    bins = np.logspace(
        np.log10(np.percentile(all_v, 1)),
        np.log10(np.percentile(all_v, 99)),
        40,
    )

    _hist_panel(axes[row, 0], tau_th_paired,
                res['tau_neu'][valid_neu_r],
                'tab:orange', f'Neurons (n={valid_neu_r.sum()})', bins)
    _hist_panel(axes[row, 1], tau_th_paired,
                res['tau_mode'][valid_mode_r],
                'tab:green', f'Eigenmodes (n={valid_mode_r.sum()})', bins)

    axes[row, 0].set_ylabel(rf'$g={g}$' + '\nDensity', fontsize=10)

for col in range(n_col):
    axes[-1, col].set_xlabel(r'$\tau_{\rm eff}$')

fig.suptitle(
    rf'Timescale distributions across $g$  —  $N={n_neurons}$  [tanh, lognormal $W$]',
    fontsize=13,
)
plt.show()

## Scatter plots across g: Theory vs Fitted

In [ ]:
fig, axes = plt.subplots(
    n_g, 2,
    figsize=(10, 3.5 * n_g),
    constrained_layout=True,
)
if n_g == 1:
    axes = axes[np.newaxis, :]

axes[0, 0].set_title('Theory vs Neuron fits (rank-rank)', fontsize=11)
axes[0, 1].set_title('Theory vs Eigenmode fits (1-to-1)', fontsize=11)

for row, res in enumerate(results):
    g            = res['g']
    tau_th_all   = res['tau_th_all']
    valid_mode_r = res['valid_mode']
    valid_neu_r  = res['valid_neu']

    tau_th_paired = tau_th_all[valid_mode_r]
    tau_mode_v    = res['tau_mode'][valid_mode_r]
    tau_neu_v     = res['tau_neu'][valid_neu_r]

    # Neuron rank-rank
    ax = axes[row, 0]
    if len(tau_th_paired) > 0 and len(tau_neu_v) > 0:
        n_min = min(len(tau_th_paired), len(tau_neu_v))
        ax.scatter(np.sort(tau_th_paired)[:n_min], np.sort(tau_neu_v)[:n_min],
                   s=8, alpha=0.3, color='tab:orange')
        v0 = min(tau_th_paired.min(), tau_neu_v.min())
        v1 = max(tau_th_paired.max(), tau_neu_v.max())
        ax.plot([v0, v1], [v0, v1], 'r--', lw=1.5)
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_ylabel(rf'$g={g}$' + '\n' + r'Fitted $\tau$', fontsize=10)
    ax.grid(True, alpha=0.3); ax.set_aspect('equal')

    # Eigenmode 1-to-1
    ax = axes[row, 1]
    if len(tau_th_paired) > 0 and len(tau_mode_v) > 0:
        ax.scatter(tau_th_paired, tau_mode_v, s=8, alpha=0.3, color='tab:green')
        v0 = min(tau_th_paired.min(), tau_mode_v.min())
        v1 = max(tau_th_paired.max(), tau_mode_v.max())
        ax.plot([v0, v1], [v0, v1], 'r--', lw=1.5)
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.grid(True, alpha=0.3); ax.set_aspect('equal')

for col in range(2):
    axes[-1, col].set_xlabel(r'Theory $\tau$')

fig.suptitle(
    rf'Theory vs fitted timescales across $g$  —  $N={n_neurons}$  [tanh, lognormal $W$]',
    fontsize=13,
)
plt.show()